# 00 — Environment Check (Phase 1)

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

This notebook proves the repository is installed correctly and that the
reproducibility core works, *before* any data is touched. Run it first in every
new Colab session.

It checks, in order:

1. clone + pinned install,
2. config loads and its fingerprint prints,
3. every RNG is pinned (demonstrated, not asserted on faith),
4. the Mubadala theme applies and renders a palette swatch,
5. the raw data is reachable and its SHA-256 hashes are recorded,
6. the Colab compute envelope is adequate.

> **Reminder:** notebooks orchestrate and visualise. All pipeline logic lives in
> `src/novafin/`. If you find yourself writing a function here, it belongs in a
> module.

## 1 · Bootstrap

`PYTHONHASHSEED` is read by CPython **at interpreter start-up**, so it is set in
the very first line — before any import — and the runtime is then restarted by
Colab only if you change it later. Everything else follows.

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
print("Running in Colab:", IN_COLAB)

In [ ]:
# --- Colab only: clone the repository and install pinned dependencies -------
REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"

if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print("Working directory:", os.getcwd())

In [ ]:
# --- Colab only: mount Drive and point the config at the course data -------
# The eight CSVs are NOT in the repository (educational-use licence).
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/01 ML in Finance/data"
    )

print("NOVAFIN_DATA_RAW =", os.environ.get("NOVAFIN_DATA_RAW", "<using config default>"))

## 2 · The standard project preamble

These five lines open **every** notebook in this project. They are the reason
two runs of the pipeline produce the same numbers.

In [ ]:
from novafin.config import load_config
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme

cfg = load_config()
logger = setup_logging("INFO", log_file=cfg.paths.logs / "00_environment_check.log")
seed_report = seed_everything(cfg.reproducibility.seed, deterministic=cfg.reproducibility.deterministic)
theme = apply_theme()

print("project          :", cfg.project.name, cfg.project.version)
print("author           :", cfg.project.author)
print("config source    :", cfg.source_path)
print("config fingerprint:", cfg.fingerprint())
print("datasets         :", sorted(cfg.datasets))
print("raw data dir     :", cfg.paths.data_raw)

### What the fingerprint is for

`cfg.fingerprint()` is a SHA-256 over the *effective* configuration. It is logged
with every MLflow run and printed in the footer of the D7 guide and the D8 deck,
so any number in any deliverable can be traced back to the exact settings that
produced it. If the fingerprint changes, the numbers are allowed to change; if it
does not, they are not.

## 3 · Prove the seeding actually works

A test that merely calls `seed_everything()` and checks it does not raise proves
nothing. The only honest check is to draw twice and compare.

In [ ]:
import numpy as np

seed_everything(42)
first = np.random.rand(5)
seed_everything(42)
second = np.random.rand(5)
seed_everything(1234)
third = np.random.rand(5)

print("same seed identical     :", np.array_equal(first, second))
print("different seed differs  :", not np.array_equal(first, third))
print()
for key, value in seed_report.as_dict().items():
    print(f"  {key:>22} : {value}")

## 4 · Theme check

The palette below is the single source of truth for every figure in the guide
and the deck. Colours come from `configs/theme.yaml`; nothing else in the
repository contains a hex code.

Contrast ratios are **measured**, not assumed — `gold #FFC72C` is 1.56:1 on
white and is therefore a fill/marker colour only; small risk text uses
`gold_text #7A5B00` at 6.32:1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from novafin.utils.theme import diverging_cmap, save_figure, swatch

fig = swatch()
plt.show()

# A quick sanity chart in the project style.
rng = np.random.default_rng(cfg.reproducibility.seed)
x = np.arange(60)
fig, ax = plt.subplots()
for label in ["Model A", "Model B", "Benchmark"]:
    ax.plot(x, rng.normal(0, 1, 60).cumsum(), label=label)
ax.set_title("Theme smoke test - cumulative series")
ax.set_xlabel("Period")
ax.set_ylabel("Cumulative value")
ax.legend()
save_figure(fig, "00_theme_smoke_test", close=False)
plt.show()

## 5 · Data reachability and provenance

This cell does **not** load the data into memory. It confirms every expected
file is present and records a SHA-256 for each, which goes into the run manifest
so results are traceable to a specific version of the inputs.

In [ ]:
from novafin.utils.io import RunManifest, sha256_file, timestamp_slug

expected = {key: ds.path(cfg.paths) for key, ds in cfg.datasets.items()}
missing = {k: p for k, p in expected.items() if not p.exists()}

if missing:
    print("MISSING FILES - set NOVAFIN_DATA_RAW to your data folder:")
    for key, path in missing.items():
        print(f"  {key:<14} expected at {path}")
else:
    print(f"{'dataset':<14} {'size (MB)':>10}  sha256[:16]")
    print("-" * 46)
    for key, path in expected.items():
        print(f"{key:<14} {path.stat().st_size / 1024**2:>10.2f}  {sha256_file(path)[:16]}")

In [ ]:
# Record a run manifest: config fingerprint + seed + data hashes + versions.
manifest = RunManifest(
    run_id=f"env-check-{timestamp_slug()}",
    config_fingerprint=cfg.fingerprint(),
    seed=cfg.reproducibility.seed,
    data_hashes=RunManifest.hash_inputs(expected.values()),
    package_versions=RunManifest.collect_versions(
        ["numpy", "pandas", "scikit-learn", "lightgbm", "xgboost",
         "optuna", "mlflow", "shap", "torch", "matplotlib"]
    ),
    notes="Phase 1 environment check.",
)
path = manifest.save(cfg.paths.artifacts / f"{manifest.run_id}.json")
print("Manifest written to:", path)
for name, version in manifest.package_versions.items():
    print(f"  {name:<14} {version}")

## 6 · Compute envelope

Phase 0 established that nothing in this project exceeds the Colab free tier:
the largest object is the 120,000 × 33 order book at roughly 64 MB, and the
whole package is about 82 MB. The GPU is needed only for the Level-3 deep
fine-tuning; `compute.use_gpu: auto` falls back to a LightGBM-only path with no
code change if no CUDA device is present.

In [ ]:
import shutil

print("GPU enabled by config :", cfg.compute.gpu_enabled())
print("Max dataframe budget  :", cfg.compute.max_dataframe_mb, "MB")
print("Free disk             :", round(shutil.disk_usage("/").free / 1024**3, 1), "GB")

try:
    import psutil

    print("Total RAM             :", round(psutil.virtual_memory().total / 1024**3, 1), "GB")
except ImportError:
    print("Total RAM             : psutil not installed (optional)")

try:
    import torch

    print("torch                 :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device                :", torch.cuda.get_device_name(0))
except ImportError:
    print("torch                 : not installed (CPU-only path is fine for Phases 1-4)")

## 7 · Leakage register reminder

Before Phase 2 touches the data, read `docs/LEAKAGE_REGISTER.md`. The three
critical entries:

| ID | Finding | Control |
|---|---|---|
| **L-01** | `Return` in the market panel is the **same-day** return (`corr = 1.000`) | Target is constructed as `Return.groupby(Ticker).shift(-1)`; same-row OHLC dropped |
| **L-02** | `Future_Return_100ms` / `Future_Price_100ms` **are** the HFT label | Both dropped from `X` |
| **L-03** | `Liquidity_Gap ≡ Expected_Outflows − Expected_Inflows` (max dev 0.01) | Forecast outflows *h* days ahead, then derive the gap |

The cell below confirms these rules are live in the config, not just written in
a document.

In [ ]:
checks = {
    "market": {"Open", "High", "Low", "Close", "Return"},
    "hft": {"Future_Return_100ms", "Future_Price_100ms"},
    "liquidity": {"Liquidity_Gap"},
}
for key, required in checks.items():
    declared = set(cfg.dataset(key).drop_always)
    status = "OK" if required <= declared else "MISSING " + str(sorted(required - declared))
    print(f"{key:<12} drop_always -> {status}")

print()
print("Forbidden features for 'loans':", sorted(cfg.dataset("loans").forbidden_features))

---

**Phase 1 complete** when every cell above runs clean.

**NEXT:** `01_setup_and_eda` — data loading, schema validation, leakage tests,
and the exploratory analysis that feeds the D7 guide and the D8 deck.